# Qwen Endpoint Smoke Test

This notebook does one thing: confirm that the local Qwen endpoint configuration works before running the longer Stargazer agent experiment.


## 1. Load the local provider configuration

The API key is only checked as present. It is never printed.


In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "traj_eval").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the traj-eval checkout.")


def clean_env(value: str | None) -> str:
    return (value or "").strip().strip('"').strip("'")


repo_root = find_repo_root(Path.cwd().resolve())
env_path = Path(os.getenv("TRAJ_EVAL_PROVIDER_ENV", repo_root / "configs" / "qwen.remote.local.env")).expanduser()
loaded = load_dotenv(env_path, override=True)

base_url = clean_env(os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE"))
api_key = clean_env(os.getenv("OPENAI_API_KEY"))
model = clean_env(os.getenv("CMBAGENT_EVAL_LOCAL_MODEL") or os.getenv("TRAJ_EVAL_MODEL") or os.getenv("OPENAI_MODEL"))
request_timeout = float(os.getenv("QWEN_REQUEST_TIMEOUT", "600"))
max_retries = int(os.getenv("QWEN_MAX_RETRIES", "10"))

setup_rows = [
    {"check": "env file exists", "value": env_path.exists()},
    {"check": "env loaded", "value": loaded},
    {"check": "base url configured", "value": bool(base_url)},
    {"check": "api key configured", "value": bool(api_key)},
    {"check": "model from env", "value": model or "auto-detect"},
    {"check": "request timeout seconds", "value": request_timeout},
    {"check": "max retries", "value": max_retries},
]
display(pd.DataFrame(setup_rows))

assert env_path.exists(), f"Missing env file: {env_path}"
assert base_url, "OPENAI_BASE_URL or OPENAI_API_BASE is missing."
assert api_key, "OPENAI_API_KEY is missing."

client = OpenAI(base_url=base_url, api_key=api_key, timeout=request_timeout, max_retries=max_retries)


,check,value
0,env file exists,True
1,env loaded,True
2,base url configured,True
3,api key configured,True
4,model from env,openai/Qwen3.5-27B-Q5_K_M.gguf
5,request timeout seconds,600.0
6,max retries,10


## 2. Select a model and send one message

This is the only API test in the notebook. The longer timeout/retry settings are shown here because the agent notebook uses the same deep-thinking provider setup.


In [2]:
models_response = client.models.list()
available_models = [item.id for item in models_response.data]
if not model and available_models:
    model = available_models[0]
    os.environ["TRAJ_EVAL_MODEL"] = model

assert model, "The provider returned no model id. Set CMBAGENT_EVAL_LOCAL_MODEL in the local env file."

chat_response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply exactly: Qwen endpoint reachable"}],
    temperature=0,
    max_tokens=80,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
message = chat_response.choices[0].message
visible_text = (message.content or getattr(message, "reasoning_content", None) or "").strip()

result_rows = [
    {"check": "models endpoint", "value": "ok", "detail": f"{len(available_models)} model(s) listed"},
    {"check": "selected model", "value": model, "detail": "from env or first /models result"},
    {"check": "chat completion", "value": bool(visible_text), "detail": visible_text},
    {"check": "finish reason", "value": chat_response.choices[0].finish_reason, "detail": "single short request"},
]
display(pd.DataFrame(result_rows))
assert visible_text, "The chat call returned no visible content."


,check,value,detail
0,models endpoint,ok,1 model(s) listed
1,selected model,openai/Qwen3.5-27B-Q5_K_M.gguf,from env or first /models result
2,chat completion,True,Qwen endpoint reachable
3,finish reason,stop,single short request


## Result

The endpoint is ready when the table above shows a selected model and a non-empty chat completion. No extra backend, streaming, or mini-agent tests are needed here.
